# Inspect Training Data Pipeline

这个 notebook 用来检查两层数据：

- 数据集本身：HDF5 / `station_metadata`
- 输入到模型前的 generator 输出：裁窗、pick、mask、PGA target

当前训练链路里，事件级裁窗是统一对整个事件做一次，所有台站共享同一个 `crop_start`，因此台站间时间仍然对齐。

In [ ]:
from pathlib import Path
import copy
import json
import sys

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path('/home/zhangb/work/people/zhangbei/team_claude/team_pytorch')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import loader_light as loader
import gemini_util_light as util

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True

In [ ]:
# 修改这里
CONFIG_PATH = ROOT / 'pga_configs' / 'transformer_japan_overfit.json'
SPLIT = 'train'   # 'train' / 'dev'
DATASET_INDEX = 0
SAMPLE_INDEX = 0
EVENT_ID = None   # 例如 '20240101123400'；None 表示按 SAMPLE_INDEX 取
OVERFIT_N = 0
MIN_STALTA_RATIO = 0.1

In [ ]:
config = json.loads(CONFIG_PATH.read_text())
training_params = copy.deepcopy(config['training_params'])
generator_params = copy.deepcopy(training_params.get('generator_params', [training_params.copy()]))
if not isinstance(training_params['data_path'], list):
    training_params['data_path'] = [training_params['data_path']]

part_map = {'train': (True, False, False), 'dev': (False, True, False)}
cache_name = 'inspect_train_ev.csv' if SPLIT == 'train' else 'inspect_dev_ev.csv'
parts = part_map[SPLIT]
overwrite_sampling_rate = training_params.get('overwrite_sampling_rate', None)

data_path = Path(training_params['data_path'][DATASET_INDEX])
if not data_path.is_absolute():
    data_path = (ROOT / data_path).resolve()
generator_param = generator_params[DATASET_INDEX]
event_metadata, _, metadata = loader.load_events(
    str(data_path),
    event_metadata_path=str(ROOT / cache_name),
    parts=parts,
    shuffle_train_dev=generator_param.get('shuffle_train_dev', False),
    custom_split=generator_param.get('custom_split', None),
    min_mag=generator_param.get('min_mag', None),
    mag_key=generator_param.get('key', 'MA'),
    overwrite_sampling_rate=overwrite_sampling_rate,
    decimate_events=generator_param.get('decimate_events', None),
    min_stalta_ratio_at_pick=MIN_STALTA_RATIO,
)

event_key = loader.detect_event_key(event_metadata.columns)
resolved_mag_key = metadata.get('resolved_mag_key', generator_param.get('key', 'MA'))
sampling_rate = metadata['sampling_rate']

print('data_path          =', data_path)
print('split              =', SPLIT)
print('rows               =', len(event_metadata))
print('events             =', event_metadata[event_key].nunique())
print('sampling_rate      =', sampling_rate)
print('resolved_mag_key   =', resolved_mag_key)
print('min_stalta_ratio   =', metadata.get('min_stalta_ratio_at_pick'))

In [ ]:
summary_cols = [c for c in [event_key, resolved_mag_key, 'Latitude', 'Longitude', 'DEPTH', 'wave_idx', 'p_pick_refined_aligned', 'p_pick_trigger_aligned', 'stalta_ratio_at_pick', 'source_network', 'sensor_class'] if c in event_metadata.columns]
display(event_metadata[summary_cols].head(10))
display(event_metadata[[resolved_mag_key, 'stalta_ratio_at_pick']].describe())

In [ ]:
if EVENT_ID is None:
    sample_row = event_metadata.iloc[SAMPLE_INDEX]
    EVENT_ID = str(sample_row[event_key])
else:
    EVENT_ID = str(EVENT_ID)

event_rows = event_metadata[event_metadata[event_key].astype(str) == EVENT_ID].copy().sort_values('wave_idx').reset_index(drop=True)
print('EVENT_ID =', EVENT_ID)
print('n_rows   =', len(event_rows))
display(event_rows[summary_cols].head(20))

In [ ]:
with h5py.File(data_path, 'r') as f:
    g = f['data'][EVENT_ID]
    wave_idx = event_rows['wave_idx'].to_numpy(dtype=int)
    waveforms = g['waveforms'][wave_idx]
    picks = g['p_picks'][wave_idx].astype(int)
    trigger_picks = g['p_pick_trigger_aligned'][wave_idx].astype(int) if 'p_pick_trigger_aligned' in g else picks.copy()
    refined_picks = g['p_pick_refined_aligned'][wave_idx].astype(int) if 'p_pick_refined_aligned' in g else picks.copy()
    ratio_at_pick = g['stalta_ratio_at_pick'][wave_idx] if 'stalta_ratio_at_pick' in g else np.full(len(wave_idx), np.nan)
    pga = g['pga'][wave_idx] if 'pga' in g else None

print('aligned event waveforms shape =', waveforms.shape)
print('p_picks range                =', int(refined_picks.min()), int(refined_picks.max()))
if pga is not None:
    print('pga looks logged?            =', bool(np.nanmax(pga) < 5 and np.nanmin(pga) > -20))

In [ ]:
# 事件级统一裁窗检查：所有台站共用同一个 crop_start
noise_seconds = generator_param.get('noise_seconds', 5)
cropped_waveforms, shifted_picks, crop_start = util._crop_aligned_event_window(
    waveforms.copy(),
    refined_picks.copy(),
    10000,
    sampling_rate,
    noise_seconds,
)
print('crop_start =', crop_start)
print('cropped shape =', cropped_waveforms.shape)
print('shifted pick min/max =', int(shifted_picks.min()), int(shifted_picks.max()))
assert np.all(shifted_picks == refined_picks - crop_start)
print('所有台站 pick 都统一减去了同一个 crop_start。')

In [ ]:
stations_to_plot = min(6, len(event_rows))
fig, axes = plt.subplots(stations_to_plot, 1, figsize=(14, 2.8 * stations_to_plot), sharex=True)
if stations_to_plot == 1:
    axes = [axes]

for i in range(stations_to_plot):
    ax = axes[i]
    ax.plot(cropped_waveforms[i, :, 2], lw=0.8, color='black', label='UD')
    ax.axvline(shifted_picks[i], color='tab:red', lw=1.2, label='refined p_pick')
    ax.axvline(trigger_picks[i] - crop_start, color='tab:blue', lw=1.0, linestyle='--', label='trigger')
    ax.set_title(f"station={event_rows.iloc[i]['station_code']}  wave_idx={int(event_rows.iloc[i]['wave_idx'])}  stalta={event_rows.iloc[i]['stalta_ratio_at_pick']:.3f}")

handles, labels = axes[0].get_legend_handles_labels()
axes[0].legend(handles[:3], labels[:3], loc='upper right')
axes[-1].set_xlabel('sample in cropped event window')
plt.tight_layout()

In [ ]:
# 构造与训练一致的 generator，并查看真正送入模型前的数据
generator_param_inspect = copy.deepcopy(generator_param)
max_stations = config['model_params']['max_stations']
n_pga_targets = config['model_params'].get('n_pga_targets', 0)
no_event_token = config['model_params'].get('no_event_token', False)
cutout = (
    sampling_rate * (noise_seconds + generator_param_inspect['cutout_start']),
    sampling_rate * (noise_seconds + generator_param_inspect['cutout_end']),
)
merged = {
    'coords_target': True,
    'label_smoothing': False,
    'station_blinding': False,
    'cutout': cutout,
    'pga_targets': n_pga_targets,
    'max_stations': max_stations,
    'sampling_rate': sampling_rate,
    'no_event_token': no_event_token,
    **generator_param_inspect,
}
dataset = util.PreloadedEventGenerator(
    event_metadata=event_metadata,
    metadata=copy.deepcopy(metadata),
    data_path=str(data_path),
    generator_params=generator_param_inspect,
    **merged,
)

dataset_event_idx = dataset.event_keys.index(EVENT_ID)
sample_pos = int(np.where(dataset.indexes == dataset_event_idx)[0][0])
inputs, outputs, p_pick_info = dataset[sample_pos]

print('waveforms shape   =', tuple(inputs[0].shape))
print('metadata shape    =', tuple(inputs[1].shape))
print('valid stations    =', int(inputs[2].sum().item()))
print('label tensors     =', [tuple(x.shape) for x in outputs])
print('p_pick_info keys  =', list(p_pick_info.keys()))

In [ ]:
wave_in = inputs[0].numpy()          # (S, C, T)
coords_in = inputs[1].numpy()
station_valid = inputs[2].numpy().astype(bool)
p_pick_raw = p_pick_info['raw'].numpy()
p_pick_shifted = p_pick_info['shifted'].numpy()
shift = float(p_pick_info['shift'].item())

valid_idx = np.where(station_valid)[0]
print('generator shift =', shift)
display(pd.DataFrame({
    'slot': np.arange(len(station_valid)),
    'station_valid': station_valid,
    'p_pick_raw': p_pick_raw,
    'p_pick_shifted': p_pick_shifted,
}).head(20))

In [ ]:
plot_slots = valid_idx[:min(6, len(valid_idx))]
fig, axes = plt.subplots(len(plot_slots), 1, figsize=(14, 2.8 * max(1, len(plot_slots))), sharex=True)
if len(plot_slots) == 1:
    axes = [axes]

for ax, slot in zip(axes, plot_slots):
    ax.plot(wave_in[slot, 2], lw=0.8, color='black', label='UD input')
    ax.axvline(p_pick_shifted[slot], color='tab:red', lw=1.2, label='shifted p_pick')
    ax.set_title(f'slot={slot}  coord={coords_in[slot]}')

handles, labels = axes[0].get_legend_handles_labels()
axes[0].legend(handles[:2], labels[:2], loc='upper right')
axes[-1].set_xlabel('sample in model input window')
plt.tight_layout()

In [ ]:
# 如果配置启用了 PGA target，这里查看目标点和标签
if len(inputs) >= 5:
    pga_targets = inputs[3].numpy()
    pga_valid = inputs[4].numpy().astype(bool)
    pga_labels = outputs[-1].numpy().reshape(-1)
    display(pd.DataFrame({
        'valid': pga_valid,
        'pga_label': pga_labels,
        'lat': pga_targets[:, 0],
        'lon': pga_targets[:, 1],
        'depth_or_height': pga_targets[:, 2],
    }).head(20))
else:
    print('当前配置没有 PGA targets。')